In [2]:
import httpx, json, time, re

# ==========================================
# 1. CẤU HÌNH (Nhớ cập nhật URL/TOKEN của Colab)
# ==========================================
URL   = "https://qui-hebrew-lifestyle-awareness.trycloudflare.com"
TOKEN = "AjVT5_bB9I7Tq-DxwX7iba0M-9-ov4gq"
MODEL = "qwen2.5:14b-instruct-q4_K_M"

# ==========================================
# 2. MOCK DATA (Tạo sẵn 6 câu test, gồm đúng & sai)
# ==========================================
MOCK_QUESTIONS = [
    {
        "id": 1, "sentence": "The manager _____ the report yesterday.",
        "A": "review", "B": "reviews", "C": "reviewed", "D": "reviewing",
        "correct": "C" # Câu đúng
    },
    {
        "id": 2, "sentence": "We recieved your application on Monday.",
        "A": "recieved", "B": "received", "C": "receiving", "D": "receive",
        "correct": "A" # Lỗi chính tả: recieved -> received (Đáp án đúng phải là B)
    },
    {
        "id": 3, "sentence": "The marketing team will launch the new campaign next Monday.",
        "A": "launch", "B": "launched", "C": "launching", "D": "launches",
        "correct": "A" # Lỗi logic: Không có chỗ trống (_____) trong câu
    },
    {
        "id": 4, "sentence": "He _____ homework every day.",
        "A": "do", "B": "did", "C": "go", "D": "went",
        "correct": "A" # Lỗi Trình bày/Đáp án: Chủ ngữ He + hiện tại đơn => thiếu 'does', A không đúng ngữ pháp
    },
    {
        "id": 5, "sentence": "All employees must submit _____ timesheets by Friday.",
        "A": "they", "B": "them", "C": "their", "D": "theirs",
        "correct": "C" # Câu đúng
    },
    {
        "id": 6, "sentence": "The company is looking _____ a candidate who can speak three languages.",
        "A": "at", "B": "for", "C": "in", "D": "on",
        "correct": "B" # Câu đúng
    }
]

# ==========================================
# 3. SYSTEM PROMPT & SCHEMA
# ==========================================
SYSTEM_PROMPT = """Bạn là chuyên gia hiệu đính đề thi TOEIC Part 5.
Hãy kiểm tra danh sách câu hỏi và tìm các lỗi sau:
1. "spelling" (Chính tả): Sai từ vựng tiếng Anh.
2. "logic" (Logic): Câu vô nghĩa, thiếu chỗ trống "_____", hoặc đáp án không khớp với chỗ trống.
3. "presentation" (Trình bày/Ngữ pháp): Lỗi ngữ pháp, sai thì, hoặc đáp án đúng (correct) bị gán sai (ví dụ: đáp án A không đúng ngữ pháp nhưng lại được set là đúng).

YÊU CẦU ĐẦU RA:
- Với mỗi câu, giữ nguyên ID.
- Trả về `has_error` (true/false).
- Trả về `error_type` (chọn "spelling", "logic", "presentation", hoặc "none").
- Trả về `explanation` (Giải thích ngắn gọn lý do sai bằng tiếng Việt).
- Trả về `original` (Bản gốc gồm sentence, A, B, C, D, correct).
- Trả về `fixed` (Bản đề xuất sửa gồm sentence, A, B, C, D, correct). Nếu không có lỗi, `fixed` giống hệt `original`.

CHỈ TRẢ VỀ JSON THEO SCHEMA."""

CHECK_SCHEMA = {
    "type": "object",
    "properties": {
        "results": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "id": {"type": "integer"},
                    "has_error": {"type": "boolean"},
                    "error_type": {"type": "string", "enum": ["spelling", "logic", "presentation", "none"]},
                    "explanation": {"type": "string"},
                    "original": {
                        "type": "object",
                        "properties": {
                            "sentence": {"type": "string"}, "A": {"type": "string"}, "B": {"type": "string"},
                            "C": {"type": "string"}, "D": {"type": "string"}, "correct": {"type": "string"}
                        },
                        "required": ["sentence", "A", "B", "C", "D", "correct"]
                    },
                    "fixed": {
                        "type": "object",
                        "properties": {
                            "sentence": {"type": "string"}, "A": {"type": "string"}, "B": {"type": "string"},
                            "C": {"type": "string"}, "D": {"type": "string"}, "correct": {"type": "string"}
                        },
                        "required": ["sentence", "A", "B", "C", "D", "correct"]
                    }
                },
                "required": ["id", "has_error", "error_type", "explanation", "original", "fixed"]
            }
        }
    },
    "required": ["results"]
}

# ==========================================
# 4. HÀM GỌI API
# ==========================================
def check_batch(client, questions_batch):
    user_prompt = "Kiểm tra và sửa các câu sau:\n" + json.dumps(questions_batch, ensure_ascii=False, indent=2)

    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        "format": CHECK_SCHEMA,
        "stream": False,
        "keep_alive": "30m",
        "options": {"num_ctx": 8192, "num_predict": 2500, "temperature": 0.0},
    }

    try:
        r = client.post(f"{URL}/api/chat", headers={"Authorization": f"Bearer {TOKEN}"}, json=payload, timeout=300)
        r.raise_for_status()
        content = r.json()["message"]["content"].strip()
        
        # Làm sạch markdown
        if content.startswith("```json"): content = content[7:]
        if content.startswith("```"): content = content[3:]
        if content.endswith("```"): content = content[:-3]
        
        return json.loads(content.strip())["results"]
    except Exception as e:
        print(f"Lỗi API: {e}")
        return []

# ==========================================
# 5. CHẠY VÀ TRÍCH XUẤT KẾT QUẢ
# ==========================================
def run():
    print(f"🔍 Bắt đầu kiểm tra {len(MOCK_QUESTIONS)} câu...\n")
    all_results = []
    
    with httpx.Client(timeout=300) as client:
        # Chia batch 5 câu / lần
        for i in range(0, len(MOCK_QUESTIONS), 5):
            batch = MOCK_QUESTIONS[i:i+5]
            results = check_batch(client, batch)
            all_results.extend(results)

    # Lọc ra các câu bị lỗi
    error_results = [r for r in all_results if r.get("has_error")]
    
    print("="*60)
    print(f"🚨 PHÁT HIỆN {len(error_results)} CÂU BỊ LỖI CẦN SỬA")
    print("="*60)
    
    # In chi tiết từng câu lỗi
    for r in error_results:
        print(f"\n▶ ID: {r['id']} | Loại lỗi: {r['error_type'].upper()}")
        print(f"  [Câu gốc]    : {r['original']['sentence']}")
        print(f"  [Đáp án gốc] : Correct = {r['original']['correct']} ({r['original']['A']} / {r['original']['B']} / {r['original']['C']} / {r['original']['D']})")
        print(f"  ❌ Lý do sai : {r['explanation']}")
        print(f"  ---------------------------")
        print(f"  [Đề xuất sửa]: {r['fixed']['sentence']}")
        print(f"  [Đáp án mới] : Correct = {r['fixed']['correct']} ({r['fixed']['A']} / {r['fixed']['B']} / {r['fixed']['C']} / {r['fixed']['D']})")
    
    # Lưu file JSON
    with open("toeic_error_proposals.json", "w", encoding="utf-8") as f:
        json.dump(error_results, f, ensure_ascii=False, indent=2)
    print("\n💾 Đã lưu danh sách câu sai và đề xuất vào: toeic_error_proposals.json")

if __name__ == "__main__":
    run()

🔍 Bắt đầu kiểm tra 6 câu...

🚨 PHÁT HIỆN 3 CÂU BỊ LỖI CẦN SỬA

▶ ID: 2 | Loại lỗi: SPELLING
  [Câu gốc]    : We recieved your application on Monday.
  [Đáp án gốc] : Correct = A (recieved / received / receiving / receive)
  ❌ Lý do sai : Sai chính tả, phải là 'received'.
  ---------------------------
  [Đề xuất sửa]: We received your application on Monday.
  [Đáp án mới] : Correct = B (recieved / received / receiving / receive)

▶ ID: 3 | Loại lỗi: LOGIC
  [Câu gốc]    : The marketing team will launch the new campaign next Monday.
  [Đáp án gốc] : Correct = A (launch / launched / launching / launches)
  ❌ Lý do sai : Đáp án A không phù hợp ngữ cảnh, nên là 'will launch'.
  ---------------------------
  [Đề xuất sửa]: The marketing team will launch the new campaign next Monday.
  [Đáp án mới] : Correct = A (will launch / launched / launching / launches)

▶ ID: 4 | Loại lỗi: LOGIC
  [Câu gốc]    : He _____ homework every day.
  [Đáp án gốc] : Correct = A (do / did / go / went)
  ❌ Lý do 